## Add AML Specific Features to Improve Model Performance

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

# Configuration
BASE_DIR = '/content/gdrive/MyDrive/AML_Competition'
FEATURE_DIR = Path(BASE_DIR) / 'features'

print("\n[1] Loading existing features...")
df = pd.read_csv(FEATURE_DIR / 'customer_features_engineered.csv')
print(f"   Loaded: {len(df):,} customers with {df.shape[1]} columns")

# Store original feature count
original_features = df.shape[1]

Mounted at /content/gdrive

[1] Loading existing features...
   Loaded: 61,410 customers with 97 columns


/tmp/ipython-input-3567872531.py:13: DtypeWarning: Columns (83,84,85,87,89) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(FEATURE_DIR / 'customer_features_engineered.csv')


In [2]:
# STRUCTURING INDICATORS
# Feature 1: Structuring Score
df['structuring_score'] = 0.0

if 'count_near_10k' in df.columns:
    # Direct count approach
    near_10k = df['count_near_10k'].fillna(0)

    # Score based on absolute count (not ratio)
    df['structuring_score'] = np.where(near_10k >= 10, 1.0,
                              np.where(near_10k >= 5, 0.7,
                              np.where(near_10k >= 3, 0.4,
                              np.where(near_10k >= 1, 0.2, 0.0))))

    # Bonus for very near threshold
    if 'count_very_near_10k' in df.columns:
        very_near = df['count_very_near_10k'].fillna(0)
        df['structuring_score'] += np.where(very_near >= 5, 0.3,
                                   np.where(very_near >= 2, 0.2,
                                   np.where(very_near >= 1, 0.1, 0.0)))

    # Bonus for round numbers
    if 'ratio_round_100' in df.columns:
        df['structuring_score'] += np.where(df['ratio_round_100'] > 0.8, 0.3,
                                   np.where(df['ratio_round_100'] > 0.6, 0.2,
                                   np.where(df['ratio_round_100'] > 0.4, 0.1, 0.0)))

    # Cap at 1.0
    df['structuring_score'] = np.minimum(df['structuring_score'], 1.0)

# Feature 2: Geographic Structuring
df['geographic_structuring'] = 0.0

if 'unique_cities' in df.columns:
    cities = df['unique_cities'].fillna(0)

    # Score based on absolute city count
    df['geographic_structuring'] = np.where(cities >= 20, 1.0,
                                   np.where(cities >= 10, 0.7,
                                   np.where(cities >= 5, 0.4,
                                   np.where(cities >= 3, 0.2, 0.0))))

# Feature 3: Temporal Structuring
df['temporal_structuring'] = 0.0

if 'night_transaction_ratio' in df.columns and 'weekend_ratio' in df.columns:
    night = df['night_transaction_ratio'].fillna(0)
    weekend = df['weekend_ratio'].fillna(0)

    # Heavy night activity = suspicious
    night_score = np.where(night > 0.5, 0.6,
                  np.where(night > 0.3, 0.4,
                  np.where(night > 0.1, 0.2, 0.0)))

    # Heavy weekend activity
    weekend_score = np.where(weekend > 0.6, 0.4,
                    np.where(weekend > 0.4, 0.2, 0.0))

    df['temporal_structuring'] = np.minimum(night_score + weekend_score, 1.0)

In [3]:
# VELOCITY INDICATORS
# Feature 4: Rapid Fund Movement
df['rapid_churn_score'] = 0.0

if 'total_inflow' in df.columns and 'total_outflow' in df.columns and 'time_span_days' in df.columns:
    total_flow = df['total_inflow'].fillna(0) + df['total_outflow'].fillna(0)
    days = np.maximum(df['time_span_days'].fillna(1), 1)

    # Daily turnover
    daily_turnover = total_flow / days

    # Score based on daily turnover amount
    df['rapid_churn_score'] = np.where(daily_turnover > 50000, 1.0,
                              np.where(daily_turnover > 20000, 0.7,
                              np.where(daily_turnover > 10000, 0.5,
                              np.where(daily_turnover > 5000, 0.3, 0.0))))

# Feature 5: Transaction Burst
df['transaction_burst_score'] = 0.0

if 'monthly_txn_cv' in df.columns:
    cv = df['monthly_txn_cv'].fillna(0)

    # High CV = very unstable patterns
    df['transaction_burst_score'] = np.where(cv > 2.0, 1.0,
                                    np.where(cv > 1.5, 0.7,
                                    np.where(cv > 1.0, 0.5,
                                    np.where(cv > 0.5, 0.3, 0.0))))


In [4]:
# INCONSISTENCY INDICATORS
# Feature 6: Income Mismatch Severity
df['income_mismatch_severity'] = 0.0

if 'spending_to_income_ratio' in df.columns:
    ratio = df['spending_to_income_ratio'].fillna(0)

    # Severe penalties for extreme mismatches
    df['income_mismatch_severity'] = np.where(ratio > 100, 1.0,
                                     np.where(ratio > 50, 0.9,
                                     np.where(ratio > 20, 0.7,
                                     np.where(ratio > 10, 0.5,
                                     np.where(ratio > 5, 0.3,
                                     np.where(ratio > 3, 0.2, 0.0))))))

# Feature 7: Business Size Mismatch
df['business_size_mismatch'] = 0.0

if 'volume_to_sales_ratio' in df.columns:
    ratio = df['volume_to_sales_ratio'].fillna(0)

    df['business_size_mismatch'] = np.where(ratio > 100, 1.0,
                                   np.where(ratio > 50, 0.8,
                                   np.where(ratio > 20, 0.6,
                                   np.where(ratio > 10, 0.4,
                                   np.where(ratio > 5, 0.2, 0.0)))))

In [5]:
# HIGH-RISK BEHAVIOR INDICATORS
# Feature 8: Cash Intensity
df['cash_intensity_score'] = 0.0

if 'cash_ratio' in df.columns:
    cash_ratio = df['cash_ratio'].fillna(0)

    # High cash usage is suspicious
    df['cash_intensity_score'] = np.where(cash_ratio > 0.8, 1.0,
                                 np.where(cash_ratio > 0.6, 0.7,
                                 np.where(cash_ratio > 0.4, 0.4,
                                 np.where(cash_ratio > 0.2, 0.2, 0.0))))

# Feature 9: International Risk
df['international_risk_score'] = 0.0

if 'international_ratio' in df.columns:
    intl_ratio = df['international_ratio'].fillna(0)

    # Base score from international ratio
    df['international_risk_score'] = np.where(intl_ratio > 0.8, 0.6,
                                     np.where(intl_ratio > 0.5, 0.4,
                                     np.where(intl_ratio > 0.2, 0.2, 0.0)))

    # Add bonus for high-risk countries
    if 'drug_country_txn_count' in df.columns:
        drug_txns = df['drug_country_txn_count'].fillna(0)
        df['international_risk_score'] += np.where(drug_txns >= 5, 0.4,
                                          np.where(drug_txns >= 2, 0.3,
                                          np.where(drug_txns >= 1, 0.2, 0.0)))

    if 'offshore_center_txn_count' in df.columns:
        offshore = df['offshore_center_txn_count'].fillna(0)
        df['international_risk_score'] += np.where(offshore >= 5, 0.3,
                                          np.where(offshore >= 2, 0.2,
                                          np.where(offshore >= 1, 0.1, 0.0)))

    # Cap at 1.0
    df['international_risk_score'] = np.minimum(df['international_risk_score'], 1.0)

# Feature 10: Channel Mixing
df['channel_mixing_score'] = 0.0

if 'channel_diversity' in df.columns:
    diversity = df['channel_diversity'].fillna(0)

    # Using many channels = obfuscation
    df['channel_mixing_score'] = np.where(diversity > 0.8, 1.0,
                                 np.where(diversity > 0.6, 0.7,
                                 np.where(diversity > 0.4, 0.4,
                                 np.where(diversity > 0.2, 0.2, 0.0))))

In [6]:
# COMPOSITE RISK SCORES
df['aml_risk_composite'] = (
    df['structuring_score'] * 4.0 +           # High weight - major red flag
    df['income_mismatch_severity'] * 3.5 +    # High weight - very suspicious
    df['business_size_mismatch'] * 2.5 +      # Medium-high weight
    df['international_risk_score'] * 3.0 +    # High weight - risky jurisdictions
    df['rapid_churn_score'] * 2.0 +           # Medium weight
    df['geographic_structuring'] * 2.0 +      # Medium weight
    df['cash_intensity_score'] * 1.5 +        # Medium-low weight
    df['temporal_structuring'] * 1.5 +        # Medium-low weight
    df['channel_mixing_score'] * 1.0 +        # Low weight
    df['transaction_burst_score'] * 1.0       # Low weight
)
# Total possible: 24.0

# Normalize to 0-1 using max value
max_possible = 24.0
df['aml_risk_composite'] = df['aml_risk_composite'] / max_possible

# Also save the raw score before normalization
df['aml_risk_composite_raw'] = df['aml_risk_composite'].copy()

# Apply power transformation to spread out the scores
# This prevents everyone clustering at low values
df['aml_risk_composite'] = df['aml_risk_composite'] ** 0.5  # Square root spreads it out

In [7]:
# INTERACTION FEATURES
# Feature 11: High-value + High-frequency
df['high_value_high_frequency'] = 0.0

if 'avg_transaction_amount' in df.columns and 'transactions_per_day' in df.columns:
    avg_amt = df['avg_transaction_amount'].fillna(0)
    freq = df['transactions_per_day'].fillna(0)

    # Both high = very suspicious
    high_amt = avg_amt > 5000
    high_freq = freq > 2

    df['high_value_high_frequency'] = np.where(high_amt & high_freq, 1.0,
                                      np.where(high_amt | high_freq, 0.5, 0.0))

# Feature 12: New customer + High activity
df['new_customer_high_activity'] = 0.0

if 'customer_tenure_days' in df.columns and 'transaction_count_total' in df.columns:
    tenure = df['customer_tenure_days'].fillna(365)
    txn_count = df['transaction_count_total'].fillna(0)

    # New customers (< 90 days) with high activity
    is_new = tenure < 90
    high_activity = txn_count > 20
    very_high_activity = txn_count > 50

    df['new_customer_high_activity'] = np.where(is_new & very_high_activity, 1.0,
                                       np.where(is_new & high_activity, 0.6, 0.0))

# Feature 13: Round amounts + Near threshold
df['round_near_threshold'] = 0.0

if 'ratio_round_100' in df.columns and 'count_near_10k' in df.columns:
    rounds = df['ratio_round_100'].fillna(0)
    near_10k = df['count_near_10k'].fillna(0)

    # Both present = very suspicious
    high_round = rounds > 0.6
    has_near_10k = near_10k >= 2

    df['round_near_threshold'] = np.where(high_round & has_near_10k, 1.0,
                                 np.where(high_round | has_near_10k, 0.5, 0.0))

In [8]:
# CLEAN UP
# Fill any NaN created
new_features = [
    'structuring_score', 'geographic_structuring', 'temporal_structuring',
    'rapid_churn_score', 'transaction_burst_score',
    'income_mismatch_severity', 'business_size_mismatch',
    'cash_intensity_score', 'international_risk_score', 'channel_mixing_score',
    'aml_risk_composite', 'aml_risk_composite_raw',
    'high_value_high_frequency', 'new_customer_high_activity', 'round_near_threshold'
]

for col in new_features:
    if col in df.columns:
        df[col] = df[col].fillna(0)
        df[col] = df[col].replace([np.inf, -np.inf], 0)

# Save enhanced features
output_path = FEATURE_DIR / 'customer_features_enhanced.csv'
df.to_csv(output_path, index=False)